# Liu2024 S-JEPA Downstream — Native 500 Hz, Option B (CUSTOM architecture)

Faithful downstream decoder for the **self-contained** Option-B S-JEPA pretraining. It reuses the
pretraining's `ConvFeatureEncoder` / `PositionalEncoder` / `SJEPABackbone` **verbatim** (NOT
braindecode), rebuilds the backbone from the checkpoint's embedded metadata, loads the pretrained
`student_backbone_state_dict`, and attaches a small classification head on the pooled
context-encoder tokens.

**Why this replaces the braindecode downstream.** The checkpoint is a custom model
(`feature_encoder.net.{0,2,3,...}` = `[Conv1d(bias)+GELU+GroupNorm]×5`, plus a transformer
`context_encoder`). braindecode's `SignalJEPA_PreLocal` has a different encoder (bias-free convs,
7 tensors), so its weights never matched — that was the `No matching feature_encoder weights` /
`First shape skip: []` error. Here the architecture matches by construction.

**Head / "PreLocal" analog.** The custom feature encoder is per-channel (no spatial conv); spatial
mixing is done by the learned spatial positional encoding + the `context_encoder` self-attention.
So the downstream forward is: `tokens, pe = backbone.tokens_pe(x)` → `ctx = context_encoder(tokens+pe)`
→ mean-pool tokens → `Linear(d_model, 2)`. `strategy="new"` freezes the whole backbone (kept in
eval() so GroupNorm/dropout stay deterministic) and trains only the head; `"full"` fine-tunes all.

**Honest expectations.** Subjects 41-50 are the SSL-held-out set, but per your decodability
permutation test most of them carry no decodable MI signal (only 44, weakly 49, show anything), so
average balanced accuracy will sit near chance for *any* method — separate "model failed" from
"these subjects have no signal" via the optional decodability cross-reference. With ~19 train / ~5
val / 16 test per split this is also high-variance. The point of this notebook is that the
pretrained weights are now loaded and used correctly; it does not manufacture signal.

## 1. Imports

In [ ]:
import os
import re
import sys
import json
import math
import random
import hashlib
import platform
from copy import deepcopy
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

from scipy.io import loadmat
import mne

# NOTE: braindecode.models.SignalJEPA intentionally NOT used — see root-cause note above.

mne.set_log_level("WARNING")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

print("Imports loaded")
print("Python:", sys.version)
print("Platform:", platform.platform())
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.metrics import (balanced_accuracy_score, accuracy_score, cohen_kappa_score,
                             precision_score, recall_score, confusion_matrix)


## 2. Downstream configuration

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    "experiment_name": "liu2024_sjepa_500hz_optionB_downstream_custom",
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-sjepa-500hz-optionB-downstream"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),

    # Checkpoint from the Option-B pretraining run.
    # Prefer the WITH-chans export: it keeps the learned pos_encoder (spatial encoding).
    "checkpoint_path": "CHANGE_ME/student_backbone_with_chans_best.pt",
    "export_variant": "with_chans",                 # with_chans (recommended) | without_chans

    # Subjects held out from SSL pretraining -> the only honest downstream set.
    "downstream_subject_ids": list(range(41, 51)),

    # Preprocessing MUST match pretraining exactly.
    "sfreq": 500.0, "source_unit": "microvolts", "final_model_unit": "microvolts",
    "reference_mode": "average", "filter_low": 0.5, "filter_high": 40.0, "filter_method": "fir",

    # Downstream MI window (Liu): 1.5-5.7 s -> 2100 samples @500 Hz.
    "mi_window_start_s": 1.5, "mi_window_s": 4.2,

    # Conv-spec fallback (only used if the checkpoint metadata lacks conv_layers_spec).
    "conv_spec_mode": "seconds_scaled_500hz", "first_kernel_s": 0.25,
    "first_stride_s_target": 1.0 / 16.0, "first_stride_rounding": "floor",
    "later_kernel_samples": 2, "later_stride_samples": 2,

    # Context transformer params — MUST match pretraining (not stored in metadata;
    # n_layers is verified against the checkpoint, nhead/dim_ff/dropout cannot be and must match).
    "context_n_layers": 4, "context_nhead": 8, "context_dim_feedforward": 256, "context_dropout": 0.0,

    # Head + training.
    "strategy": "new",                              # new = freeze backbone, train head | full = fine-tune all
    "pool": "mean",                                 # mean | max
    "batch_size": 8, "n_epochs": 300, "early_stopping_patience": 40,
    "learning_rate": 3e-4, "weight_decay": 1e-4, "val_fraction": 0.2, "label_smoothing": 0.0,

    # Liu within-subject evaluation.
    "n_repeats": 10, "test_size": 0.4, "split_random_state": 2026,

    "require_full_backbone_load": True,
    "decodability_csv_path": None,                  # optional: decodability_per_subject.csv
    "device": "auto", "seed": 2026,
}
print("Strategy:", CONFIG["strategy"], "| export:", CONFIG["export_variant"],
      "| downstream subjects:", CONFIG["downstream_subject_ids"])


## 3. Run setup + seconds geometry (carried verbatim from pretraining)

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def resolve_device(requested="auto"):
    requested = str(requested).lower()
    if requested == "cpu":
        return torch.device("cpu")
    if requested == "cuda":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def create_run_id(config):
    stamp = datetime.now().strftime("%Y%m%d_%H%M")
    digest = hashlib.md5(json.dumps(config, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{stamp}_500hz_{digest}"


def make_seconds_scaled_conv_spec(config):
    """Create a 500 Hz conv spec whose real-time behavior matches S-JEPA.

    Default S-JEPA at 128 Hz uses:
      (8, 32, 8), then four (kernel=2, stride=2) layers.
    That means:
      first kernel = 32/128 = 0.25 s
      total stride = 8*2*2*2*2 = 128 samples = 1.0 s
      receptive field = 152/128 = 1.1875 s

    At 500 Hz, we use first kernel ~= 125 samples and first stride ~= 31 samples.
    Total stride = 31*16 = 496 samples = 0.992 s.
    Receptive field = 590 samples = 1.18 s.
    """
    sfreq = float(config["sfreq"])
    k1 = int(round(float(config["first_kernel_s"]) * sfreq))
    raw_stride = float(config["first_stride_s_target"]) * sfreq
    if config.get("first_stride_rounding", "floor") == "floor":
        s1 = int(math.floor(raw_stride))
    elif config.get("first_stride_rounding") == "ceil":
        s1 = int(math.ceil(raw_stride))
    else:
        s1 = int(round(raw_stride))
    s1 = max(1, s1)
    k_later = int(config["later_kernel_samples"])
    s_later = int(config["later_stride_samples"])
    spec = (
        (8, k1, s1),
        (16, k_later, s_later),
        (32, k_later, s_later),
        (64, k_later, s_later),
        (64, k_later, s_later),
    )
    return spec


def conv_geometry(conv_spec, n_times, sfreq):
    out = int(n_times)
    total_stride = 1
    receptive = 1
    running_stride = 1
    for _, kernel, stride in conv_spec:
        out = (out - int(kernel)) // int(stride) + 1
        receptive = receptive + (int(kernel) - 1) * running_stride
        running_stride *= int(stride)
        total_stride *= int(stride)
    return {
        "n_tokens_per_channel": int(out),
        "total_stride_samples": int(total_stride),
        "token_sfreq_hz": float(sfreq) / float(total_stride),
        "token_stride_s": float(total_stride) / float(sfreq),
        "receptive_field_samples": int(receptive),
        "receptive_field_s": float(receptive) / float(sfreq),
    }


set_seed(CONFIG["seed"])
DEVICE = resolve_device(CONFIG["device"])
RUN_ID = create_run_id(CONFIG)
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
print("Artifact dir:", ARTIFACT_DIR, "| Device:", DEVICE)


## 4. Channel metadata + finite coordinates (carried verbatim)

In [ ]:
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17  # CPz source reference in Liu2024 source MAT
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]
CH_NAMES = [SOURCE_EEG_CHANNEL_NAMES_30[i] for i in SOURCE_EEG_CHANNEL_INDICES_29]
MONTAGE_ALIAS = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}

def build_channel_coordinates(ch_names):
    """Return (chs_info, xyz_meters, ch_pos_norm, audit_df). Raises on any missing/degenerate coord."""
    montage = mne.channels.make_standard_montage("standard_1020")
    ch_pos = montage.get_positions()["ch_pos"]
    info = mne.create_info(ch_names=list(ch_names), sfreq=float(CONFIG["sfreq"]), ch_types="eeg")
    xyz = np.zeros((len(ch_names), 3), dtype=np.float64)
    used = []
    for k, ch in enumerate(info["chs"]):
        name = ch["ch_name"]
        lookup = MONTAGE_ALIAS.get(name, name)
        if lookup not in ch_pos:
            raise RuntimeError(f"Missing montage coordinate for '{name}' (lookup '{lookup}'). "
                               f"Add an alias in MONTAGE_ALIAS.")
        pos = np.asarray(ch_pos[lookup], dtype=np.float64)
        if not np.isfinite(pos).all():
            raise RuntimeError(f"Non-finite montage coordinate for '{name}'.")
        loc = np.zeros(12, dtype=float); loc[:3] = pos
        ch["loc"][:] = loc
        xyz[k] = pos
        used.append(lookup)

    # finite / non-degenerate checks
    if not np.isfinite(xyz).all():
        raise RuntimeError("Non-finite channel coordinates after assembly.")
    axis_range = xyz.max(0) - xyz.min(0)
    if np.any(axis_range <= 1e-8):
        raise RuntimeError(f"Degenerate coordinate axis (range={axis_range}). Montage not 3-D.")
    # duplicate-position guard (two channels at the same point would break spatial encoding)
    from scipy.spatial.distance import pdist
    if pdist(xyz).min() <= 1e-9:
        raise RuntimeError("Two channels share identical coordinates; check names/aliases.")

    # normalized coordinates for the positional encoder: center + scale to unit ball, eps-safe
    center = xyz.mean(0)
    centered = xyz - center
    radius = float(np.linalg.norm(centered, axis=1).max())
    ch_pos_norm = (centered / (radius + 1e-8)).astype(np.float32)
    if not np.isfinite(ch_pos_norm).all():
        raise RuntimeError("Non-finite normalized coordinates (should be impossible with eps).")

    audit = pd.DataFrame({
        "ch_name": list(ch_names), "montage_lookup": used,
        "x_m": xyz[:, 0].round(4), "y_m": xyz[:, 1].round(4), "z_m": xyz[:, 2].round(4),
        "nx": ch_pos_norm[:, 0].round(3), "ny": ch_pos_norm[:, 1].round(3), "nz": ch_pos_norm[:, 2].round(3),
        "norm": np.linalg.norm(ch_pos_norm, axis=1).round(3),
    })
    return info["chs"], xyz, ch_pos_norm, audit

CHS_INFO, CH_XYZ_M, CH_POS_NORM, COORD_AUDIT = build_channel_coordinates(CH_NAMES)
CH_POSITIONS = torch.tensor(CH_XYZ_M.astype(np.float32))        # meters, for mask-sampler distances
CH_POS_NORM_T = torch.tensor(CH_POS_NORM)                        # normalized, for the pos encoder

print(f"Channels: {len(CH_NAMES)}")
print(CH_NAMES)
print("\nCoordinate audit (normalized coords are finite, |norm|<=1):")
try:
    from IPython.display import display; display(COORD_AUDIT)
except Exception:
    print(COORD_AUDIT.to_string(index=False))
assert np.isfinite(CH_POS_NORM).all()
print(f"\nAll {len(CH_NAMES)} channel coordinates finite; max |norm| = {np.linalg.norm(CH_POS_NORM,axis=1).max():.3f}")


## 5. Load + preprocess (carried verbatim; float64-correct filtering)

In [ ]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []


def subject_id_from_path(path):
    text = str(path)
    match = re.search(r"sub[-_ ]?(\d{1,2})", text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from {path}")


def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")


def _walk_mat_object(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray) and obj.dtype == object:
        if obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        else:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")


def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if any(token in lname for token in ["rawdata", "raw", "data"]):
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    return score


def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    score = 0
    if "label" in lname or "class" in lname:
        score += 10
    if flat.size in (39, 40):
        score += 5
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    if unique and unique.issubset({"0", "1", "2", "1.0", "2.0", "0.0"}):
        score += 3
    return score


def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []
    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]
    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)
    if arr.shape[1] < 30 or arr.shape[2] < 3000:
        raise ValueError(f"Could not normalize to trials x channels x samples, got {arr.shape}")
    return arr


def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    raw_candidates = []
    label_candidates = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            arr = np.asarray(value)
            if arr.ndim == 3:
                raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
            flat = arr.ravel()
            if flat.size in (39, 40):
                label_candidates.append((_score_label_candidate(name, arr), name, arr))
    if not raw_candidates:
        raise RuntimeError(f"No rawdata candidate found in {path}")
    raw_candidates.sort(key=lambda x: x[0], reverse=True)
    label_candidates.sort(key=lambda x: x[0], reverse=True)
    raw_name, raw = raw_candidates[0][1], raw_candidates[0][2]
    if not label_candidates:
        raise RuntimeError(f"No label candidate found in {path}")
    label_name, labels = label_candidates[0][1], label_candidates[0][2]
    raw = _normalize_rawdata_shape(raw, labels)
    labels = np.asarray(labels).ravel().astype(int)
    if labels.min() == 1:
        labels = labels - 1
    if raw.shape[0] != labels.size:
        raise RuntimeError(f"Trial/label mismatch for {path}: raw={raw.shape}, labels={labels.shape}")
    return raw, labels.astype(np.int64), raw_name, label_name


def preprocess_subject(rawdata, labels, subject_id):
    """Return preprocessed full 8 s trials as microvolts, shape trials x 29 x 4000.

    MNE FIR filtering expects float64 input in recent versions.  Keep the
    filtering/reference path in float64, then cast back to float32 only after
    preprocessing so the training tensors remain compact.
    """
    X = np.asarray(rawdata[:, SOURCE_EEG_CHANNEL_INDICES_29, :], dtype=np.float64)
    y = np.asarray(labels, dtype=np.int64)

    # Convert to volts for MNE filtering, average-reference before filter, then return microvolts.
    X_volts = X * 1e-6 if CONFIG["source_unit"] == "microvolts" else X
    X_volts = np.asarray(X_volts, dtype=np.float64, order="C")
    if CONFIG["reference_mode"] == "average":
        X_volts = X_volts - X_volts.mean(axis=1, keepdims=True)
        X_volts = np.asarray(X_volts, dtype=np.float64, order="C")
    X_volts = mne.filter.filter_data(
        X_volts,
        sfreq=float(CONFIG["sfreq"]),
        l_freq=float(CONFIG["filter_low"]),
        h_freq=float(CONFIG["filter_high"]),
        method=CONFIG["filter_method"],
        phase="zero",
        fir_design="firwin",
        verbose=False,
    )
    X_uv = X_volts * 1e6 if CONFIG["final_model_unit"] == "microvolts" else X_volts
    return X_uv.astype(np.float32, copy=False), y


def make_pretraining_windows(X_full_trials, y):
    sfreq = float(CONFIG["sfreq"])
    n_times = X_full_trials.shape[-1]
    mode = CONFIG["pretrain_window_mode"]
    windows = []
    labels = []

    if mode == "full_trial":
        start = 0
        length = n_times
        windows = [X_full_trials]
        labels = [y]
    elif mode == "fixed_crop":
        start = int(round(float(CONFIG["pretrain_start_s"]) * sfreq))
        length = int(round(float(CONFIG["pretrain_window_s"]) * sfreq))
        stop = start + length
        if stop > n_times:
            raise RuntimeError(f"fixed_crop [{start}:{stop}] exceeds trial length {n_times}")
        windows = [X_full_trials[:, :, start:stop]]
        labels = [y]
    elif mode == "sliding":
        length = int(round(float(CONFIG["pretrain_window_s"]) * sfreq))
        start0 = int(round(float(CONFIG["pretrain_start_s"]) * sfreq))
        stop_limit = int(round(float(CONFIG["sliding_stop_s"]) * sfreq))
        stride = int(round(float(CONFIG["sliding_stride_s"]) * sfreq))
        for start in range(start0, stop_limit - length + 1, stride):
            windows.append(X_full_trials[:, :, start:start + length])
            labels.append(y)
    else:
        raise ValueError("pretrain_window_mode must be full_trial, fixed_crop, or sliding")

    Xw = np.concatenate(windows, axis=0).astype(np.float32)
    yw = np.concatenate(labels, axis=0).astype(np.int64)
    return Xw, yw


class ArrayDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.zeros(len(self.X), dtype=np.int64) if y is None else np.asarray(y, dtype=np.int64)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), int(self.y[idx])

## 6. Self-contained S-JEPA classes (carried VERBATIM from pretraining cell 14)

In [ ]:
def assert_finite(name, tensor):
    if not torch.isfinite(tensor).all():
        bad = int((~torch.isfinite(tensor)).sum().item())
        raise RuntimeError(f"Non-finite tensor detected: {name}; count={bad}")


class ConvFeatureEncoder(nn.Module):
    """Per-channel temporal conv stack (S-JEPA local encoder). (B,C,T) -> (B,C,n_tok,D)."""
    def __init__(self, conv_spec):
        super().__init__()
        blocks, in_ch = [], 1
        for out_ch, k, s in conv_spec:
            blocks += [nn.Conv1d(in_ch, int(out_ch), kernel_size=int(k), stride=int(s)),
                       nn.GELU(), nn.GroupNorm(1, int(out_ch))]   # GroupNorm: batch-size-independent, stable
            in_ch = int(out_ch)
        self.net = nn.Sequential(*blocks); self.out_dim = in_ch
    def forward(self, x):
        B, C, T = x.shape
        h = self.net(x.reshape(B * C, 1, T))         # (B*C, D, n_tok)
        D, n_tok = h.shape[1], h.shape[-1]
        return h.permute(0, 2, 1).reshape(B, C, n_tok, D)


class PositionalEncoder(nn.Module):
    """Finite-by-construction PE: spatial = MLP(normalized xyz); temporal = bounded sinusoid."""
    def __init__(self, d_model, ch_pos_norm, max_tokens=8192):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even for sinusoidal PE"
        self.d_model = d_model
        self.register_buffer("ch_pos", torch.as_tensor(ch_pos_norm, dtype=torch.float32))
        self.spatial_mlp = nn.Sequential(nn.Linear(3, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        pe = torch.zeros(max_tokens, d_model)
        pos = torch.arange(max_tokens, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("temporal_pe", pe)
    def spatial(self):                 # (C, D)
        return self.spatial_mlp(self.ch_pos)
    def temporal(self, n_tok):         # (n_tok, D)
        return self.temporal_pe[:n_tok]
    def forward(self, n_chans, n_tok):  # (C*n_tok, D), channel-major to match feature flattening
        sp = self.spatial()                       # (C, D)
        tp = self.temporal(n_tok)                 # (n_tok, D)
        return (sp[:, None, :] + tp[None, :, :]).reshape(n_chans * n_tok, self.d_model)


class SJEPABackbone(nn.Module):
    """feature_encoder -> (+ positional) -> context_encoder. EMA-copied to form the target encoder."""
    def __init__(self, conv_spec, ch_pos_norm, n_layers, nhead, dim_ff, dropout):
        super().__init__()
        self.feature_encoder = ConvFeatureEncoder(conv_spec)
        D = self.feature_encoder.out_dim
        self.pos_encoder = PositionalEncoder(D, ch_pos_norm)
        layer = nn.TransformerEncoderLayer(d_model=D, nhead=nhead, dim_feedforward=dim_ff,
                                           dropout=dropout, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.context_encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.d_model = D
    def tokens_pe(self, x):
        local = self.feature_encoder(x)                 # (B,C,n_tok,D)
        B, C, n_tok, D = local.shape
        pe = self.pos_encoder(C, n_tok)                 # (C*n_tok, D)
        return local.reshape(B, C * n_tok, D), pe, (B, C, n_tok, D)


## 7. Build the downstream dataset on subjects 41-50 (MI window)

Same loader/preprocessing as pretraining (full 8 s trials), then cropped to the Liu MI window
(1.5-5.7 s = 2100 samples). The conv encoder is fully convolutional, so a shorter window simply
yields fewer tokens/channel than pretraining's 8 s windows — the model is token-count-agnostic.

In [ ]:
def crop_mi_window(X_full):
    """X_full: trials x 29 x 4000 -> trials x 29 x 2100 (1.5-5.7 s @500 Hz)."""
    s = int(round(float(CONFIG["mi_window_start_s"]) * float(CONFIG["sfreq"])))
    n = int(round(float(CONFIG["mi_window_s"]) * float(CONFIG["sfreq"])))
    stop = s + n
    if stop > X_full.shape[-1]:
        raise RuntimeError(f"MI window [{s}:{stop}] exceeds trial length {X_full.shape[-1]}")
    return X_full[:, :, s:stop]

paths = {subject_id_from_path(p): p for p in find_source_mat_files(CONFIG["source_extract_dir"])}
SUBJECT_DATA, rows = {}, []
for sid in CONFIG["downstream_subject_ids"]:
    if int(sid) not in paths:
        raise FileNotFoundError(f"No .mat for subject {sid} under {CONFIG['source_extract_dir']}")
    raw, labels, raw_name, label_name = load_subject_mat(paths[int(sid)])
    X_full, y = preprocess_subject(raw, labels, sid)
    X = crop_mi_window(X_full).astype(np.float32)
    SUBJECT_DATA[str(sid)] = (X, y.astype(np.int64))
    rows.append({"subject_id": int(sid), "X_shape": list(X.shape),
                 "class_0": int((y == 0).sum()), "class_1": int((y == 1).sum())})
WINDOW_SAMPLES = int(round(float(CONFIG["mi_window_s"]) * float(CONFIG["sfreq"])))
inventory = pd.DataFrame(rows)
try:
    from IPython.display import display; display(inventory)
except Exception:
    print(inventory.to_string(index=False))
assert all(r["class_0"] > 1 and r["class_1"] > 1 for r in rows), "A subject is missing a class."
print("MI window samples:", WINDOW_SAMPLES)


## 8. Rebuild the backbone from the checkpoint and load the pretrained weights

The conv spec comes from the checkpoint's embedded metadata (so the geometry is guaranteed to match
pretraining). `n_layers` is read back from the state-dict and used to override CONFIG if they
disagree. The load asserts that the **entire feature_encoder and context_encoder** are present —
a partial/silent load is treated as a hard error.

In [ ]:
ckpt = torch.load(CONFIG["checkpoint_path"], map_location="cpu", weights_only=False)
if "student_backbone_state_dict" not in ckpt:
    raise RuntimeError("Checkpoint has no 'student_backbone_state_dict'. Use a student_backbone_*.pt export.")
sd = ckpt["student_backbone_state_dict"]
meta = ckpt.get("backbone_export_metadata", ckpt)   # student_backbone_* payloads spread meta at top level

# conv spec from metadata if available, else rebuild from CONFIG
if isinstance(meta, dict) and "conv_layers_spec" in meta:
    conv_spec = tuple(tuple(int(v) for v in layer) for layer in meta["conv_layers_spec"])
else:
    conv_spec = make_seconds_scaled_conv_spec(CONFIG)
print("Conv spec:", conv_spec)

# verify n_layers against the checkpoint
layer_ids = sorted({int(k.split("context_encoder.layers.")[1].split(".")[0])
                    for k in sd if "context_encoder.layers." in k})
n_layers_ckpt = (max(layer_ids) + 1) if layer_ids else int(CONFIG["context_n_layers"])
if n_layers_ckpt != int(CONFIG["context_n_layers"]):
    print(f"[warn] context_n_layers {CONFIG['context_n_layers']} -> {n_layers_ckpt} (from checkpoint)")

BACKBONE = SJEPABackbone(conv_spec, CH_POS_NORM,
                         n_layers=n_layers_ckpt, nhead=int(CONFIG["context_nhead"]),
                         dim_ff=int(CONFIG["context_dim_feedforward"]),
                         dropout=float(CONFIG["context_dropout"])).to(DEVICE)

result = BACKBONE.load_state_dict(sd, strict=False)
model_keys, ckpt_keys = set(BACKBONE.state_dict()), set(sd)
missing = sorted(model_keys - ckpt_keys)        # in model, absent from checkpoint
unexpected = sorted(ckpt_keys - model_keys)      # in checkpoint, absent from model
core_missing = [k for k in missing if k.startswith(("feature_encoder.", "context_encoder."))]
pos_missing  = [k for k in missing if k.startswith("pos_encoder.")]

if CONFIG["require_full_backbone_load"] and core_missing:
    raise RuntimeError(
        f"{len(core_missing)} feature/context params missing from the checkpoint, e.g. {core_missing[:6]}. "
        "Conv geometry or context params (nhead/dim_ff/n_layers) likely differ from pretraining."
    )
if unexpected:
    raise RuntimeError(f"Checkpoint has {len(unexpected)} keys not in the model, e.g. {unexpected[:6]}. "
                       "Architecture mismatch — wrong checkpoint or wrong context params.")
if pos_missing:
    print(f"[note] {len(pos_missing)} pos_encoder params absent (export_variant='without_chans'). "
          "Spatial positional encoding is at INIT (random spatial_mlp) -> expect degraded transfer. "
          "Use student_backbone_with_chans_best.pt for full fidelity.")

print(json.dumps({
    "checkpoint": CONFIG["checkpoint_path"],
    "loaded": len(model_keys & ckpt_keys), "model_params": len(model_keys),
    "missing": len(missing), "unexpected": len(unexpected),
    "feature_context_fully_loaded": (len(core_missing) == 0),
    "pos_encoder_loaded": (len(pos_missing) == 0),
    "d_model": int(BACKBONE.d_model), "n_layers": n_layers_ckpt,
}, indent=2))


## 9. Downstream classifier + train / evaluate one split

In [ ]:
class SJEPADownstream(nn.Module):
    """Pretrained backbone -> context tokens -> pool -> linear head."""
    def __init__(self, backbone, n_outputs=2, pool="mean"):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = nn.Linear(backbone.d_model, n_outputs)
    def forward(self, x):
        tokens, pe, _ = self.backbone.tokens_pe(x)             # (B, C*n_tok, D), (C*n_tok, D)
        ctx = self.backbone.context_encoder(tokens + pe.unsqueeze(0))   # (B, C*n_tok, D)
        pooled = ctx.mean(dim=1) if self.pool == "mean" else ctx.max(dim=1).values
        return self.head(pooled)


def set_trainable(model, strategy):
    strategy = str(strategy).lower()
    for p in model.parameters():
        p.requires_grad = False
    if strategy == "new":
        for p in model.head.parameters():
            p.requires_grad = True
    elif strategy == "full":
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f"strategy must be 'new' or 'full', got {strategy}")
    return int(sum(p.requires_grad for p in model.parameters()))


def set_phase_train_mode(model, strategy):
    """Head in train mode; frozen backbone stays in eval() so GroupNorm/dropout are deterministic."""
    model.train()
    if str(strategy).lower() == "new":
        model.backbone.eval()


def make_downstream_model():
    # "new": reuse the shared FROZEN backbone (never updated -> no leakage), fresh head per split.
    # "full": deepcopy so each split fine-tunes from the pretrained weights independently.
    bb = BACKBONE if CONFIG["strategy"] == "new" else deepcopy(BACKBONE)
    model = SJEPADownstream(bb, n_outputs=2, pool=CONFIG["pool"]).to(DEVICE)
    set_trainable(model, CONFIG["strategy"])
    return model


def train_one_model(model, train_idx, test_idx, X, y, split_seed):
    Xtr, ytr = X[train_idx], y[train_idx]
    tr_i, val_i = train_test_split(np.arange(len(train_idx)), test_size=CONFIG["val_fraction"],
                                   stratify=ytr, random_state=split_seed)
    train_loader = DataLoader(ArrayDataset(Xtr[tr_i], ytr[tr_i]), batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader   = DataLoader(ArrayDataset(Xtr[val_i], ytr[val_i]), batch_size=CONFIG["batch_size"])
    test_loader  = DataLoader(ArrayDataset(X[test_idx], y[test_idx]), batch_size=CONFIG["batch_size"])

    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                           lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    crit = nn.CrossEntropyLoss(label_smoothing=float(CONFIG["label_smoothing"]))

    best_val, best_state, patience = float("inf"), None, 0
    for epoch in range(CONFIG["n_epochs"]):
        set_phase_train_mode(model, CONFIG["strategy"])
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        vloss, n = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vloss += crit(model(xb), yb).item() * len(yb); n += len(yb)
        vloss /= max(n, 1)
        if vloss < best_val - 1e-5:
            best_val, patience = vloss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= CONFIG["early_stopping_patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
            gts.append(yb.numpy())
    return np.concatenate(gts), np.concatenate(preds)


## 10. Liu-style within-subject evaluation (subjects 41-50)

Repeated stratified 60/40 splits (24 train / 16 test), validation carved from train only, fresh
head per split with a per-split seed.

In [ ]:
def make_subject_splits(y, sid):
    sss = StratifiedShuffleSplit(n_splits=CONFIG["n_repeats"], test_size=CONFIG["test_size"],
                                 random_state=CONFIG["split_random_state"] + int(sid))
    return list(sss.split(np.zeros(len(y)), y))

cv_rows, load_infos = [], []
for sid in CONFIG["downstream_subject_ids"]:
    X, y = SUBJECT_DATA[str(sid)]
    for split_id, (train_idx, test_idx) in enumerate(make_subject_splits(y, sid), start=1):
        set_seed(int(CONFIG["seed"]) + 1000 * int(sid) + split_id)   # reproducible head init per split
        model = make_downstream_model()
        yt, yp = train_one_model(model, train_idx, test_idx, X, y,
                                 split_seed=CONFIG["split_random_state"] + 1000 * int(sid) + split_id)
        cv_rows.append({
            "subject_id": int(sid), "split": split_id,
            "balanced_accuracy": balanced_accuracy_score(yt, yp),
            "accuracy": accuracy_score(yt, yp),
            "kappa": cohen_kappa_score(yt, yp),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0),
        })
    print(f"subject {sid}: done")

cv = pd.DataFrame(cv_rows)
cv.to_csv(ARTIFACT_DIR / "cv_results.csv", index=False)

summary_by_subject = cv.groupby("subject_id").agg(
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    balanced_accuracy_std=("balanced_accuracy", "std"),
    accuracy_mean=("accuracy", "mean"),
    kappa_mean=("kappa", "mean"),
).reset_index()
summary_by_subject.to_csv(ARTIFACT_DIR / "subject_summary.csv", index=False)

global_summary = {
    "strategy": CONFIG["strategy"], "export_variant": CONFIG["export_variant"],
    "n_subjects": len(CONFIG["downstream_subject_ids"]),
    "mean_balanced_accuracy": float(cv["balanced_accuracy"].mean()),
    "std_balanced_accuracy": float(cv["balanced_accuracy"].std()),
    "mean_accuracy": float(cv["accuracy"].mean()),
    "mean_kappa": float(cv["kappa"].mean()),
}
with open(ARTIFACT_DIR / "global_summary.json", "w") as f:
    json.dump(global_summary, f, indent=2)
print(json.dumps(global_summary, indent=2))

# optional decodability cross-reference (subjects 41-50 are mostly non-decodable per the permutation test)
dec_path = CONFIG.get("decodability_csv_path")
if dec_path and Path(dec_path).is_file():
    dec = pd.read_csv(dec_path)
    keep = [c for c in ["subject_id", "honest_balacc_8_30", "fgmdm_balacc_8_30", "decodable"] if c in dec.columns]
    if "subject_id" in keep:
        summary_by_subject = summary_by_subject.merge(dec[keep], on="subject_id", how="left")
        summary_by_subject.to_csv(ARTIFACT_DIR / "subject_summary.csv", index=False)
        if "decodable" in summary_by_subject.columns:
            print("Decodable subjects within 41-50:",
                  summary_by_subject.loc[summary_by_subject["decodable"] == True, "subject_id"].tolist())


## 11. Per-subject plot

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
if "decodable" in summary_by_subject.columns:
    colors = ["#2c7fb8" if bool(d) else "#bdbdbd" for d in summary_by_subject["decodable"].fillna(False)]
else:
    colors = "#2c7fb8"
ax.bar(summary_by_subject["subject_id"].astype(str), summary_by_subject["balanced_accuracy_mean"], color=colors)
ax.axhline(0.5, linestyle="--", linewidth=1, color="grey")
ax.set_ylim(0, 1); ax.set_ylabel("balanced accuracy"); ax.set_xlabel("subject")
ax.set_title(f"Custom S-JEPA downstream ({CONFIG['strategy']}) — subjects 41-50")
fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "subject_balanced_accuracy.png", dpi=160); plt.show()
print("Saved:", ARTIFACT_DIR / "subject_balanced_accuracy.png")


## 12. Notes

- This is the faithful downstream for the **custom** Option-B S-JEPA: the pretrained
  feature_encoder **and** context_encoder are loaded and used; the head reads pooled context tokens.
  The load asserts full feature/context coverage, so a silent partial load can't happen.
- Use `student_backbone_with_chans_best.pt` (`export_variant="with_chans"`) to keep the learned
  spatial positional encoding. The `without_chans` export drops `pos_encoder` (spatial MLP at init)
  and will be weaker.
- `strategy="new"` (frozen backbone) is the honest transfer test; `"full"` will almost certainly
  overfit on ~19 training trials. Set `decodability_csv_path` to flag which of 41-50 actually carry
  signal — average accuracy near chance is expected because most of them don't.
- Preprocessing here matches pretraining (500 Hz, average ref, 0.5-40 FIR, float64 filtering). The
  MI window is 1.5-5.7 s; the conv encoder yields fewer tokens/channel than pretraining's 8 s
  windows, which is fine (the model is token-count-agnostic).